# Skill Demand Forecasting - EDA

## Objective
Understand the skill demand time series data to build an appropriate forecasting model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Data

In [ ]:
# Load skill demand data
df = pd.read_csv('backend/database/seed/skill_demand.tsv', sep='\t')

# Convert date columns
df['period_start'] = pd.to_datetime(df['period_start'])
df['period_end'] = pd.to_datetime(df['period_end'])

print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDate range: {df['period_start'].min()} to {df['period_start'].max()}")
print(f"\nFirst few rows:")
print(df.head(10))

## 2. Data Overview

In [ ]:
# Basic statistics
print(f"Unique skills: {df['skill_name'].nunique()}")
print(f"Unique periods: {df['period_start'].nunique()}")
print(f"\nDemand statistics:")
print(df['demand_count'].describe())

# Check for missing values
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Top 20 skills by average demand
top_skills = df.groupby('skill_name')['demand_count'].agg(['mean', 'max', 'min', 'std', 'count'])
top_skills = top_skills.sort_values('mean', ascending=False)
print("Top 20 skills by average demand:")
print(top_skills.head(20))

## 3. Time Series Visualization

In [ ]:
# Prepare data for plotting - pivot by skill
pivot_df = df.pivot_table(index='period_start', columns='skill_name', values='demand_count', fill_value=0)
print(f"Pivot shape: {pivot_df.shape}")
print(f"\nPivot sample:")
print(pivot_df.head())

In [ ]:
# Plot top 8 skills
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

top_8 = top_skills.head(8).index.tolist()

for idx, skill in enumerate(top_8):
    ax = axes[idx]
    data = pivot_df[skill]
    ax.plot(data.index, data.values, marker='o', linewidth=2, markersize=6, color='steelblue')
    ax.set_title(f'{skill}\n(avg={data.mean():.1f}, max={data.max():.0f})', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylabel('Demand Count')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig('eda_top_8_skills.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Top 8 skills plot saved")

## 4. Time Series Characteristics

In [ ]:
# Analyze key characteristics for each top skill
def analyze_skill_ts(skill_name, data):
    """
    Analyze time series characteristics:
    - Trend (linear fit)
    - Volatility (std dev)
    - Outliers (values >3 std from mean)
    - Seasonality (check for month-to-month patterns)
    """
    from scipy import stats
    
    # Remove zeros for better analysis
    nonzero = data[data > 0]
    if len(nonzero) < 3:
        return None
    
    # Trend (linear regression on indices)
    x = np.arange(len(data))
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, data.values)
    
    # Volatility
    volatility = data.std() / data.mean() if data.mean() > 0 else 0
    
    # Outliers (values beyond 3 std from mean)
    mean = data.mean()
    std = data.std()
    outliers = ((data > mean + 3*std) | (data < mean - 3*std)).sum()
    
    return {
        'skill': skill_name,
        'mean': data.mean(),
        'std': data.std(),
        'min': data.min(),
        'max': data.max(),
        'trend_slope': slope,
        'trend_r2': r_value**2,
        'volatility': volatility,
        'outliers_count': outliers,
        'length': len(data)
    }

# Analyze top 15 skills
analyses = []
for skill in top_skills.head(15).index:
    result = analyze_skill_ts(skill, pivot_df[skill])
    if result:
        analyses.append(result)

analysis_df = pd.DataFrame(analyses)
print("Time Series Characteristics (Top 15 skills):")
print(analysis_df.to_string())

## 5. Trend Analysis

In [ ]:
# Visualize trends
fig, ax = plt.subplots(figsize=(12, 8))

for skill in top_skills.head(5).index:
    data = pivot_df[skill]
    ax.plot(data.index, data.values, marker='o', label=skill, linewidth=2, markersize=5)
    
    # Add trend line
    x = np.arange(len(data))
    z = np.polyfit(x, data.values, 1)
    p = np.poly1d(z)
    ax.plot(data.index, p(x), '--', alpha=0.6, linewidth=2)

ax.set_title('Top 5 Skills - Trends and Patterns', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Demand Count')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
plt.tight_layout()
plt.savefig('eda_trends.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Trends plot saved")

## 6. Volatility and Variability

In [ ]:
# Volatility comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Volatility vs Mean
ax = axes[0]
volatilities = analysis_df.sort_values('volatility')
colors = ['red' if v > 0.5 else 'orange' if v > 0.3 else 'green' for v in volatilities['volatility']]
ax.barh(volatilities['skill'], volatilities['volatility'], color=colors)
ax.set_xlabel('Volatility (std/mean)')
ax.set_title('Skill Demand Volatility', fontweight='bold')
ax.axvline(x=0.3, color='orange', linestyle='--', alpha=0.5, label='High volatility threshold')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

# Plot 2: Distribution of demand counts
ax = axes[1]
all_demands = df['demand_count'].values
ax.hist(all_demands, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(x=np.mean(all_demands), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(all_demands):.1f}')
ax.axvline(x=np.median(all_demands), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(all_demands):.1f}')
ax.set_xlabel('Demand Count')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Skill Demand', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('eda_volatility.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Volatility plot saved")

## 7. Outlier Detection

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = (data < lower_bound) | (data > upper_bound)
    return outliers, lower_bound, upper_bound

outlier_counts = {}
for skill in top_skills.head(15).index:
    data = pivot_df[skill]
    outliers, lower, upper = detect_outliers_iqr(data)
    outlier_counts[skill] = outliers.sum()

outlier_df = pd.DataFrame(list(outlier_counts.items()), columns=['skill', 'outlier_count'])
outlier_df = outlier_df.sort_values('outlier_count', ascending=False)

print("Outlier counts (IQR method):")
print(outlier_df.to_string())

## 8. Data Quality Check

In [ ]:
# Check data completeness per skill
skill_periods = df.groupby('skill_name').size()
expected_periods = df['period_start'].nunique()

print(f"Total unique periods: {expected_periods}")
print(f"\nSkills with {expected_periods} periods (complete data):")
complete = skill_periods[skill_periods == expected_periods]
print(f"  {len(complete)} out of {len(skill_periods)} skills ({100*len(complete)/len(skill_periods):.1f}%)")

# Skills with sparse data
print(f"\nSkills with sparse data (<10 periods):")
sparse = skill_periods[skill_periods < 10]
if len(sparse) > 0:
    print(sparse.sort_values())

## 9. Seasonality Analysis

In [ ]:
# Check for month-of-year seasonality
df_with_month = df.copy()
df_with_month['month'] = df_with_month['period_start'].dt.month

monthly_pattern = df_with_month.groupby('month')['demand_count'].agg(['mean', 'std', 'count'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(monthly_pattern.index, monthly_pattern['mean'], color='steelblue', alpha=0.7, label='Average Demand')
ax.errorbar(monthly_pattern.index, monthly_pattern['mean'], yerr=monthly_pattern['std'], 
            fmt='none', color='red', capsize=5, label='Std Dev')
ax.set_xlabel('Month of Year')
ax.set_ylabel('Average Demand Count')
ax.set_title('Seasonality: Average Demand by Month', fontweight='bold')
ax.set_xticks(range(1, 13))
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('eda_seasonality.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Seasonality plot saved")

print("\nMonthly pattern:")
print(monthly_pattern)

## 10. Key Insights & Modeling Recommendations

In [ ]:
print("="*70)
print("KEY INSIGHTS FOR MODELING")
print("="*70)

print(f"\n1. DATA OVERVIEW:")
print(f"   - Time range: {df['period_start'].min().date()} to {df['period_start'].max().date()}")
print(f"   - Duration: {(df['period_start'].max() - df['period_start'].min()).days / 30:.0f} months")
print(f"   - Unique skills: {df['skill_name'].nunique()}")
print(f"   - Total data points: {len(df)}")

print(f"\n2. DEMAND RANGE:")
print(f"   - Min: {df['demand_count'].min()}")
print(f"   - Max: {df['demand_count'].max()}")
print(f"   - Mean: {df['demand_count'].mean():.2f}")
print(f"   - Median: {df['demand_count'].median():.2f}")

print(f"\n3. VOLATILITY:")
high_vol = (analysis_df['volatility'] > 0.5).sum()
med_vol = ((analysis_df['volatility'] > 0.3) & (analysis_df['volatility'] <= 0.5)).sum()
low_vol = (analysis_df['volatility'] <= 0.3).sum()
print(f"   - High volatility (>0.5): {high_vol} skills")
print(f"   - Medium volatility (0.3-0.5): {med_vol} skills")
print(f"   - Low volatility (<0.3): {low_vol} skills")

print(f"\n4. OUTLIERS:")
total_outliers = outlier_df['outlier_count'].sum()
print(f"   - Total outliers (IQR method): {total_outliers} out of {len(df)} points")
print(f"   - Skills affected: {(outlier_df['outlier_count'] > 0).sum()}")

print(f"\n5. TRENDS:")
increasing = (analysis_df['trend_slope'] > 0.05).sum()
decreasing = (analysis_df['trend_slope'] < -0.05).sum()
stable = ((analysis_df['trend_slope'] >= -0.05) & (analysis_df['trend_slope'] <= 0.05)).sum()
print(f"   - Increasing trend: {increasing} skills")
print(f"   - Stable trend: {stable} skills")
print(f"   - Decreasing trend: {decreasing} skills")

print(f"\n6. DATA COMPLETENESS:")
print(f"   - Complete series (all {expected_periods} months): {len(complete)} skills")
print(f"   - Sparse data (<10 months): {len(sparse)} skills")

print(f"\n7. SEASONALITY:")
max_seasonal = monthly_pattern['mean'].max()
min_seasonal = monthly_pattern['mean'].min()
seasonal_strength = (max_seasonal - min_seasonal) / monthly_pattern['mean'].mean() * 100
print(f"   - Seasonal variation: {seasonal_strength:.1f}% of mean")
print(f"   - Peak month: {monthly_pattern['mean'].idxmax()}")
print(f"   - Trough month: {monthly_pattern['mean'].idxmin()}")

print(f"\n" + "="*70)
print("MODELING RECOMMENDATIONS:")
print("="*70)
print("""
✓ Use LSTM model:
  - Can capture complex temporal patterns
  - Handles variable-length sequences
  - Good for skills with trends and seasonality

✓ Data preprocessing:
  - Normalize/scale data (MinMaxScaler to [0,1])
  - Handle outliers with IQR method before training
  - Fill sparse skills or exclude those with <6 months data

✓ Model architecture:
  - Lookback window: 6 months (captures seasonality + trend)
  - LSTM layer: 32-64 units with dropout (0.2)
  - Dense layers: 16 units + final output layer
  - Optimizer: Adam
  - Loss: MSE (regression task)

✓ Training strategy:
  - Per-skill models (allows skill-specific tuning)
  - 50-100 epochs with early stopping
  - Validation split: 20% of historical data

✓ Forecast generation:
  - Recursive multi-step forecasting (predict 1 month ahead, use as input for next)
  - Denormalize forecasts using fitted scaler
  - Clip negatives to 0
  - Generate confidence intervals from historical volatility
""")

## 11. Summary Statistics Table

In [ ]:
# Create summary table
summary = analysis_df[['skill', 'mean', 'std', 'min', 'max', 'trend_slope', 'volatility', 'outliers_count']].copy()
summary = summary.round(2)
summary = summary.sort_values('mean', ascending=False)

print("\nComplete Summary for Top 15 Skills:")
print(summary.to_string(index=False))

# Save to CSV
summary.to_csv('eda_summary.csv', index=False)
print("\n✓ Summary saved to eda_summary.csv")

## 12. Next Steps

Based on this EDA, proceed with:
1. Data preprocessing (normalize, handle outliers)
2. LSTM model development (train per-skill)
3. Forecast generation (32 months into future)
4. Validation and error metrics
5. Load results into database